In [70]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk 
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /home/akyrr/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/akyrr/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/akyrr/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to /home/akyrr/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/akyrr/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [71]:
df = pd.read_csv('Call Of Duty.csv')

In [72]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3882 entries, 0 to 3881
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   reviews  3882 non-null   str  
 1   ratings  3882 non-null   int64
dtypes: int64(1), str(1)
memory usage: 60.8 KB


In [73]:
df.head()

,reviews,ratings
0,I've been playing for years now and it's been ...,4
1,An annoying experience all round. Ever since t...,1
2,"I love the game though. It's close to real, bu...",3
3,I've been playing for years now and it's been ...,4
4,An annoying experience all round. Ever since t...,1


In [74]:
stemmer = WordNetLemmatizer()
def preprocess_text(text):
    text = text.lower()
    tokens = nltk.word_tokenize(text)
    tokens = [word for word in tokens if word.isalpha()]
    tokens = [word for word in tokens if word not in stopwords.words('english')]
    tokens = [stemmer.lemmatize(word) for word in tokens]
    tokens = re.sub(r'\b\w{1,2}\b', '', ' '.join(tokens)).split()
    tokens = re.sub(r'\s+', ' ', ' '.join(tokens)).split()
    tokens = re.sub(r'\b\d+\b', '', ' '.join(tokens)).split()
    return ' '.join(tokens)

df['Processed_Review'] = df['reviews'].apply(preprocess_text)

In [75]:
df['Processed_Review'].head(10)

0    playing year quite addictive developer plss fi...
1    annoying experience round ever since update se...
2    love game though close real greatest problem c...
3    playing year quite addictive developer plss fi...
4    annoying experience round ever since update se...
5    cod happens best mobile game update becomes le...
6    codm awesome guarantee best shooting game ever...
7    easy star would give star nope based reality i...
8    wonderful concept graphic gameplay outta inter...
9    simply love game graphic good never boring tha...
Name: Processed_Review, dtype: str

In [76]:
def build_vocab(text):
    all_text = ' '.join(text).split()
    vocab = set(all_text)
    
    vocab_idx = {word: idx for idx, word in enumerate(vocab)}
    return vocab_idx

In [77]:
def TF(text_srs, vocab):
    n_doc = len(text_srs)
    n_vocab = len(vocab)

    tf_matrix = np.zeros((n_doc, n_vocab))

    for row, text in enumerate(text_srs):
        words = text.split()
        for w in words:
            if w in vocab:
                col = vocab[w]
                tf_matrix[row, col] += 1
    return tf_matrix

In [78]:
def IDF(text_srs, vocab):
    n_doc = len(text_srs)
    vocab_list = list(vocab.keys())

    df_count = dict.fromkeys(vocab_list, 0)
    for text in text_srs:
        unique_word = set(text.split())
        for word in unique_word:
            if word in vocab:
                df_count[word] += 1
    
    idf_value = {}
    for word, count in df_count.items():
        idf_value[word] = np.log(n_doc / (1 + count))

    return idf_value

In [79]:
def TF_IDF(tf_matrix, vocab_idx, idf_values):
    tfidf_matrix = tf_matrix.copy()

    n_doc, n_vocab = tf_matrix.shape

    for word, col_idx in vocab_idx.items():
        idf_score = idf_values[word]
        tfidf_matrix[:, col_idx] = tfidf_matrix[:, col_idx] * idf_score

    return tfidf_matrix

In [80]:
df.head()

,reviews,ratings,Processed_Review
0,I've been playing for years now and it's been ...,4,playing year quite addictive developer plss fi...
1,An annoying experience all round. Ever since t...,1,annoying experience round ever since update se...
2,"I love the game though. It's close to real, bu...",3,love game though close real greatest problem c...
3,I've been playing for years now and it's been ...,4,playing year quite addictive developer plss fi...
4,An annoying experience all round. Ever since t...,1,annoying experience round ever since update se...


In [81]:
X = df['Processed_Review']
y = df['ratings']

In [82]:
all_words = set()
for doc in X:
    all_words.update(doc.split())

word_list = list(all_words)
word_to_idx = {word: idx for idx, word in enumerate(word_list)}

num_docs = len(X)
vocab_size = len(word_list)

# traain
tf_matrix = np.zeros((num_docs, vocab_size))
for i, doc in enumerate(X):
    words = doc.split()
    word_counts = np.bincount([word_to_idx[word] for word in words if word in word_to_idx], minlength=vocab_size)
    tf_matrix[i] = word_counts / len(words) if len(words) > 0 else 0

doc_freq = np.sum(tf_matrix > 0, axis=0)
idf = np.log(num_docs / (1 + doc_freq))

tfidf = tf_matrix * idf

# # tes
# tfidf_test = np.zeros((len(X), vocab_size))
# for i, doc in enumerate(X):
#     words = doc.split()
#     word_counts = np.bincount([word_to_idx[word] for word in words if word in word_to_idx], minlength=vocab_size)
#     tfidf_test[i] = (word_counts / len(words) if len(words) > 0 else 0) * idf

print(f'TF-IDF Train shape: {tfidf.shape}')
# print(f'TF-IDF Test shape: {tfidf_test.shape}')

TF-IDF Train shape: (3882, 6764)
TF-IDF Test shape: (3882, 6764)


In [91]:
tfidf_df = pd.DataFrame(tfidf)
tfidf_df.head()

,0,1,2,3,4,5,6,7,8,9,...,6754,6755,6756,6757,6758,6759,6760,6761,6762,6763
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [99]:
df['Processed_Review'][10]

'reminds old cod option use remote control bluetooth linkage mobile device amazing accurate use phone mirror app play big screen one minor complaint hook controller sometimes opponent controller even though game boast match others also using controller little unfair hour still star game'

In [100]:
df['reviews'][10]

"Reminds me of the old CoD. The option to use a PS5/XBO remote control for Bluetooth linkage to your mobile device is amazing. It's accurate and I can use my phones mirror app to play on the big screen. I have only one minor complaint.. When you hook up a controller, sometimes my opponents do not have controllers even though the game boasts that it will match you with others who are also using controllers. It's a little unfair during off-peak hours. Still a 5 star game!"

# gjadi


In [ ]:
# vocab_index = build_vocab(X)

# idf_values = IDF(X, vocab_index)

# tf = TF(X, vocab_index)

# X_matrix = TF_IDF(tf, vocab_index, idf_values)

# print(X_matrix.shape)

(3882, 6764)


In [ ]:
tfidf_2 = pd.DataFrame(X_matrix)

In [ ]:
# tfidf.head()